# A — BVNR Benchmark Analytics

This notebook drives the `bvnr_bench` CLI tool, collects its JSON
output, and visualises parsing throughput across profiles and payload
sizes.  **No `libbvnr_shared.so` dependency** — only the compiled
binary is required.

## Contents
1. [Run the benchmark](#1.-Run-the-benchmark)
2. [Throughput bar chart](#2.-Throughput-bar-chart)
3. [Throughput scaling](#3.-Throughput-scaling)
4. [Wall vs. CPU time](#4.-Wall-vs.-CPU-time)
5. [Latency heatmap](#5.-Latency-heatmap)
6. [Minimum-overhead mode](#6.-Minimum-overhead-mode)


## Setup


In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

from bovnar.analytics import (
    run_benchmark,
    benchmark_df,
    plot_throughput_bars,
    plot_size_scaling,
    plot_wall_vs_cpu,
    plot_latency_heatmap,
    ALL_PROFILES,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110


## 1. Run the benchmark

Adjust `BINARY` to point at your compiled `bvnr_bench` executable.
The `--json` flag is added automatically by `run_benchmark`.


In [ ]:
BINARY = Path('../../build/bvnr_bench')  # adjust as needed

records = run_benchmark(
    BINARY,
    profiles=ALL_PROFILES,
    sizes=(1024, 4096, 16384, 65536),
    iterations=100,
    warmup=10,
)

df = benchmark_df(records)
print(f"{len(df)} result rows across {df['profile'].nunique()} profiles")
df.head()


## 2. Throughput bar chart

One group per payload size, one bar per profile.  Log scale is used
because throughput spans roughly two orders of magnitude.


In [ ]:
fig = plot_throughput_bars(df, metric="mb_per_sec", log_scale=True)
plt.show()


The same chart for assignments/s shows the relative cost of the
callback dispatch vs. raw byte throughput:


In [ ]:
fig = plot_throughput_bars(df, metric="ass_per_sec", log_scale=True)
plt.show()


## 3. Throughput scaling

Line chart on a log₂ x-axis makes cache-saturation "knees" visible:
a flat line means the parser is CPU-bound regardless of payload size;
a rising line indicates the parser is benefiting from cache warmth at
small sizes.


In [ ]:
fig = plot_size_scaling(df, metric="mb_per_sec")
plt.show()


## 4. Wall vs. CPU time

Points on the dashed diagonal are purely CPU-bound.  Points above
it indicate OS scheduling or memory-bus pressure.  Point size scales
with payload size.


In [ ]:
fig = plot_wall_vs_cpu(df)
plt.show()


## 5. Latency heatmap

Per-iteration wall time in µs, indexed by profile × payload size.
Annotated cells make it easy to spot outliers at a glance.


In [ ]:
fig = plot_latency_heatmap(df)
plt.show()


## 6. Minimum-overhead mode

`--min-overhead` disables the `on_verified` callback, giving pure
lexer throughput.  Comparing with and without reveals the fraction
of time spent inside the callback dispatch path.


In [ ]:
records_mo = run_benchmark(
    BINARY,
    profiles=('scalars', 'typed', 'units'),
    sizes=(4096, 16384),
    iterations=100,
    min_overhead=True,
)
df_mo = benchmark_df(records_mo)

import pandas as pd

df['mode']    = 'with callback'
df_mo['mode'] = 'min overhead'

combined = pd.concat([df, df_mo], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 4))
for mode, grp in combined.groupby('mode'):
    means = grp.groupby('profile', observed=True)['mb_per_sec'].mean()
    ax.bar(
        [f"{p}\n{mode}" for p in means.index],
        means.values,
        label=mode,
        alpha=0.8,
    )
ax.set_ylabel('Throughput (MB/s)')
ax.set_title('Callback overhead — with vs. without on_verified')
ax.set_yscale("log")
fig.tight_layout()
plt.show()
